# Tarefa 2 - Base inicial com Selenium

Este notebook e apenas uma base para voce comecar a Tarefa 2 do Laboratorio 1.

A ideia aqui e:
- coletar a lista principal do IMDb Top 250;
- abrir paginas de filmes em nova aba;
- deixar uma estrutura pronta para voce completar poster, nota, generos e diretores.

Ao longo do notebook, cada etapa foi comentada com a origem no material da professora.


## Onde esta a base teorica

- Enunciado da tarefa: `Laboratorio_1.pdf`.
- Por que IMDb precisa de Selenium: `AULA 11.pdf` e `Orientacoes_Tarefa1.ipynb`.
- Selenium na pratica: `Aula12-Selenium.ipynb`.
- Seletores CSS, `find` e `find_all`: `AULA 10.pdf`.


In [ ]:
# Fonte desta etapa:
# - Aula12-Selenium.ipynb, celulas 1 e 2
# - Aula 12 - Prática com Selenium.pdf, pagina 2
# Aqui estao os imports basicos usados pela professora nos exemplos com Selenium.
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import json
import re
import time


## Configuracao inicial

Use um limite pequeno no inicio para testar mais rapido. Quando tudo estiver funcionando, troque para 250.


In [ ]:
URL_LISTA = "https://www.imdb.com/chart/top/"
LIMITE_LISTA = 10
LIMITE_DETALHES = 3
ARQUIVO_JSON = "imdb_top250_base.json"

print("URL da lista:", URL_LISTA)
print("Quantidade para teste da lista:", LIMITE_LISTA)
print("Quantidade de paginas individuais para teste:", LIMITE_DETALHES)


In [ ]:
# Fonte desta etapa:
# - Aula12-Selenium.ipynb, celulas 3 e 4
# - AULA 11.pdf, paginas 12 a 14 (ideia de WebDriver)
# Inicializacao do navegador no mesmo estilo mostrado pela professora.
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 10)


## Etapa 1 - Coletar os cards da lista principal

A estrutura abaixo segue bem de perto o exemplo de IMDb visto na notebook da professora.

Fonte principal desta etapa:
- Aula12-Selenium.ipynb, celula 20
- AULA 11.pdf, paginas 9 a 12 para a justificativa de usar Selenium no IMDb


In [ ]:
# Fonte desta etapa:
# - Aula12-Selenium.ipynb, celula 20
# A professora usa wait.until(...) + CSS_SELECTOR para esperar os cards do IMDb.
driver.get(URL_LISTA)

filmes = wait.until(
    EC.presence_of_all_elements_located((
        By.CSS_SELECTOR,
        "li.ipc-metadata-list-summary-item"
    ))
)

print("Quantidade de cards encontrados:", len(filmes))


## Etapa 2 - Extrair os dados basicos da lista

Aqui voce ja pega uma base muito importante da tarefa:
- titulo
- ano
- nota do card
- url da pagina do filme

Isso te ajuda a completar o item 1 do enunciado antes de partir para os detalhes da pagina individual.

Fonte principal desta etapa:
- Aula12-Selenium.ipynb, celula 20
- AULA 10.pdf, paginas 4 a 11 para revisar busca e seletores
- AULA 11.pdf, pagina 27 para os tipos de localizadores do Selenium


In [ ]:
# Fonte desta etapa:
# - Aula12-Selenium.ipynb, celula 20
# Aqui a estrutura foi mantida propositalmente parecida com o exemplo da professora:
# titulo por h3, ano via regex nos metadados, nota no card e URL pelo link.
dados_lista = []

for i, filme in enumerate(filmes[:LIMITE_LISTA], start=1):
    try:
        titulo = filme.find_element(By.CSS_SELECTOR, "h3").text.strip()
    except:
        titulo = None

    meta_items = filme.find_elements(By.CSS_SELECTOR, "li.ipc-inline-list__item")
    ano = None

    for item in meta_items:
        texto = item.text.strip()
        if re.fullmatch(r"\d{4}", texto):
            ano = texto
            break

    try:
        nota = filme.find_element(By.CSS_SELECTOR, "span.ipc-rating-star").text.strip()
    except:
        nota = None

    try:
        url_filme = filme.find_element(By.TAG_NAME, "a").get_attribute("href")
    except:
        url_filme = None

    dados_lista.append({
        "rank": i,
        "titulo_lista": titulo,
        "ano_lista": ano,
        "nota_lista": nota,
        "url_filme": url_filme
    })

    print(f"{i}. {titulo} | ano={ano} | nota={nota}")

print("\nTotal coletado na lista:", len(dados_lista))


## Etapa 3 - Funcao base para visitar a pagina do filme

Aqui esta a parte mais importante para voce desenvolver a habilidade.

A funcao abaixo:
- abre a pagina do filme em nova aba;
- espera o `<h1>` aparecer;
- devolve uma estrutura pronta para voce completar.

Os campos `url_poster`, `imagem_poster`, `nota_imdb`, `generos` e `diretores` ficaram com `TODO` de proposito.

Fonte principal desta etapa:
- Aula12-Selenium.ipynb, celulas 22 a 25
- AULA 11.pdf, paginas 27, 30 e 31 para localizacao de elementos


In [ ]:
# Fonte desta etapa:
# - Aula12-Selenium.ipynb, celula 23 para abrir o filme e esperar o h1
# - Aula12-Selenium.ipynb, celula 25 para a estrategia de nova aba
# Esta funcao nao fecha a tarefa por completo: ela so deixa a estrutura pronta
# para voce inspecionar a pagina e completar os seletores que faltam.
def extrair_detalhes_filme(url_filme):
    dados = {
        "titulo": None,
        "ano": None,
        "url_poster": None,
        "imagem_poster": None,
        "nota_imdb": None,
        "generos": [],
        "diretores": [],
        "url_filme": url_filme
    }

    if url_filme is None:
        return dados

    aba_principal = driver.current_window_handle
    driver.execute_script("window.open(arguments[0]);", url_filme)
    driver.switch_to.window(driver.window_handles[-1])

    try:
        titulo_pagina = wait.until(
            EC.presence_of_element_located((By.TAG_NAME, "h1"))
        ).text.strip()
        dados["titulo"] = titulo_pagina

        print("Pagina aberta com sucesso:", titulo_pagina)

        # TODO 1:
        # Fonte para estudar como localizar isso:
        # - AULA 11.pdf, pagina 27 (find_element)
        # - AULA 11.pdf, paginas 30 e 31 (XPath)
        # - AULA 10.pdf, paginas 6 a 11 (seletores CSS)
        # Inspecione o poster no navegador e descubra um seletor bom para pegar:
        # - url_poster
        # - imagem_poster

        # TODO 2:
        # Fonte para estudar como localizar isso:
        # - Aula12-Selenium.ipynb, celula 20 (nota e ano em outro contexto IMDb)
        # - AULA 11.pdf, pagina 27
        # Inspecione a area de metadados da pagina para completar:
        # - ano
        # - nota_imdb

        # TODO 3:
        # Fonte para estudar como localizar isso:
        # - AULA 10.pdf, paginas 6 a 11
        # - AULA 11.pdf, paginas 30 e 31
        # Inspecione a area de chips/links de genero para preencher a lista generos.

        # TODO 4:
        # Fonte para estudar como localizar isso:
        # - AULA 11.pdf, pagina 27
        # - AULA 11.pdf, paginas 30 e 31
        # Inspecione a secao de creditos para preencher a lista diretores.

    except Exception as erro:
        print("Erro ao abrir pagina do filme:", erro)

    finally:
        driver.close()
        driver.switch_to.window(aba_principal)

    return dados


## Etapa 4 - Testar a navegacao nas paginas individuais

Comece com poucas paginas. Quando a estrutura estiver pronta, aumente o limite.

Fonte principal desta etapa:
- Aula12-Selenium.ipynb, celulas 23 e 25


In [ ]:
# Fonte desta etapa:
# - Aula12-Selenium.ipynb, celulas 23 e 25
# A logica aqui e testar poucas paginas individuais antes de subir para 250.
dados_detalhados = []

for item in dados_lista[:LIMITE_DETALHES]:
    print("\nVisitando filme:", item["titulo_lista"])
    detalhes = extrair_detalhes_filme(item["url_filme"])

    # Junta os dados da lista com os dados detalhados.
    # Isso facilita bastante a organizacao final do JSON.
    filme_final = {**item, **detalhes}
    dados_detalhados.append(filme_final)

    time.sleep(1)

print("\nQuantidade de filmes detalhados:", len(dados_detalhados))
print(json.dumps(dados_detalhados, indent=2, ensure_ascii=False))


## Etapa 5 - Salvar o JSON base

Depois que voce completar os TODOs, este mesmo trecho pode virar o arquivo final da tarefa.

Fonte principal desta etapa:
- Aula12-Selenium.ipynb, celulas 17, 18 e 21


In [ ]:
# Fonte desta etapa:
# - Aula12-Selenium.ipynb, celulas 17, 18 e 21
# A professora mostra a ideia de montar lista de dicionarios e salvar em JSON.
with open(ARQUIVO_JSON, "w", encoding="utf-8") as arquivo:
    json.dump(dados_detalhados, arquivo, indent=2, ensure_ascii=False)

print("Arquivo JSON salvo com sucesso:", ARQUIVO_JSON)


## Encerramento

Feche o navegador ao final da execucao.


In [ ]:
# Fonte desta etapa:
# - Aula12-Selenium.ipynb, celulas 11, 14, 16, 18, 23 e 25
# Em todos os exemplos, a professora encerra com driver.quit().
driver.quit()
print("Navegador fechado.")
